In [150]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
from scipy.stats import kruskal

In [151]:
df1 = pd.read_csv("EternalReturn_kakaogames_2024.csv", encoding="utf-8")


In [152]:
df1.shape

(204425, 228)

In [153]:
# null값 확인
df1.isnull().sum()[df1.isnull().sum() > 0]

nickname                 3
language                37
killer                7601
killDetail            7608
causeOfDeath         21268
placeOfDeath          7602
killerCharacter       8863
killerWeapon          9718
killer2              64909
killDetail2          64909
causeOfDeath2        76383
placeOfDeath2        64906
killerCharacter2     66102
killerWeapon2        66685
killer3             123667
killDetail3         123667
causeOfDeath3       130934
placeOfDeath3       123667
killerCharacter3    124405
killerWeapon3       124718
dtype: int64

In [154]:
# 닉네임 결측치 세 행만 제거한다면 24명이 들어가야 하는 하나의 게임에 영향을 미치므로
# 결측치가 있는 게임(gameId) 자체를 삭제
#bad_game_ids = df[df['nickname'].isnull()]['gameId'].unique()
#df = df[~df['gameId'].isin(bad_game_ids)]

# 조언 받아 'user1369612'으로 대체
df1['nickname'] = df1['nickname'].fillna('user1369612')

df1.shape

(204425, 228)

In [155]:
# 중복 데이터 제거
# 한 게임에 동일 유저가 두 명 들어가면 안 됨.
# userNum으로 해서 상관 없긴 한데, 이터널리턴은 다행히 중복닉 안 된다고 하네요
# gameId와 userNum의 조합 유니크하게

# 변화 없음

df1 = df1.drop_duplicates(subset=['gameId', 'userNum'])
df1.shape

(204425, 228)

In [156]:
# 1일 낮: 190초 / 1일 밤: 130초
# 플탐 1분 미만인 튕긴/초반 탈주 데이터 제거 : 8개 제거
# 과연 지워도 됨? 나중에 탈주 데이터 사용할 것임!
# 1. playTime이 60초 미만인 '삭제 대상' 데이터만 추출
deleted_rows = df1[df1['playTime'] < 60]

# 3. (옵션) 특정 컬럼만 집중적으로 확인하고 싶을 때
print(deleted_rows[['characterLevel', 'gameRank', 'gameId', 'giveUp','playerDeaths','IsLeavingBeforeCreditRevivalTerminate','userNum', 'nickname', 'playTime', 'characterNum']])

# 정상적인 탈주/이탈 데이터라고 간주하고 삭제하지 않음.
df1.shape

        characterLevel  gameRank    gameId  giveUp  playerDeaths  \
204391               1         7  35380762       0             1   
204392               1         7  35380762       0             1   
204393               1         7  35380762       0             1   
204409               1         7  35572109       0             1   
204410               1         7  35572109       0             1   
204411               1         8  35587156       1             1   
204415               1         7  35685351       1             1   
204423               1         8  35804415       1             1   

        IsLeavingBeforeCreditRevivalTerminate  userNum  nickname  playTime  \
204391                                   True  4908991    超级超级胖胖        32   
204392                                   True  4908945      超级胖胖        32   
204393                                   True  4908882  sakura57        34   
204409                                   True  4917396       DYC        37 

(204425, 228)

In [157]:
# 킬은 있는데 데미지가 0인 경우 제거 : 1개 제거
# 이거도 지워도 됨?
yes_kill_no_damage_mask = (df1['playerKill'] > 0) & (df1['damageToPlayer'] == 0)
removed_df = df1[yes_kill_no_damage_mask]
df1 = df1[~yes_kill_no_damage_mask]
removed_df[
    [
        'killDetails',
        'damageToPlayer',
        'damageToPlayer_trap',
        'damageToPlayer_basic',
        'damageToPlayer_skill',
        'damageToPlayer_itemSkill',
        'damageToPlayer_direct',
    ]
]
# 적에게 피격당한 교전상태에서 중립 동물//중립 딜로 죽음 가능성 높음.
# 데이터 일단 그대로 사용하기로 결정.

,killDetails,damageToPlayer,damageToPlayer_trap,damageToPlayer_basic,damageToPlayer_skill,damageToPlayer_itemSkill,damageToPlayer_direct
193167,"{""60"":1}",0,0,0,0,0,0


In [158]:
# 모수 확인
total_unique_users = df1['userNum'].nunique()
total_unique_users

# 20만 데이터 중에 9만 명의 순수 유저가 있음을 확인

92705

In [159]:
# 큐 타입, 3쿼드가 듀오보다 많음
df1["preMade"].value_counts()

preMade
1    96569
3    60794
2    44610
4     2451
Name: count, dtype: int64

In [160]:
# 매칭 모드 : 3-일반 맵, 4- 코발트 프로토콜
df1["matchingTeamMode"].value_counts()

matchingTeamMode
3    192736
4     11688
Name: count, dtype: int64

In [161]:
#랭크포인트 이상치 조회
invalid_mask = pd.to_numeric(df1['rankPoint'], errors='coerce').isna() & df1['rankPoint'].notna()
df_invalid = df1[invalid_mask]

df_invalid[['rankPoint']].head()

,rankPoint


In [162]:
# 티어 별 세그먼트 분류
bins = [-1, 2400, 3600, 6400,np.inf]
labels = ['아이언&브론즈', '실버&골드', '플래티넘&다이아', '메테오라이트+']

df1['rank_group'] = pd.cut(
    df1['rankPoint'],
    bins=bins,
    labels=labels
)

In [163]:
# 티어 별 세그먼트 분포 
# 랭크게임에서 여부 판별 시 반드시 랭크게임인지 구분 필요
df1['rank_group'].value_counts().sort_index()

rank_group
아이언&브론즈     143129
실버&골드        19356
플래티넘&다이아     37313
메테오라이트+       4626
Name: count, dtype: int64

In [164]:
# 1. 우선 게임 단위로 
# mmr 수치 중에 하나라도 mmr이 0이 아니라면
game_rank_flag = (
    df1
    .groupby("gameId")
    .apply(
        lambda x: (
            (x["mmrGainInGame"] != 0).any() or
            (x["mmrLossEntryCost"] != 0).any()
        )
    )
    .reset_index(name="is_rank_game")
)

# 통째로 df1에 merge
df1 = df1.merge(game_rank_flag, on="gameId", how="left")

# 랭크게임 여부를 확인하는 rank_type_v2 컬럼 추가
df1["rank_type_v2"] = df1["is_rank_game"].map(
    {True: "rank_game", False: "rank_none"}
)

df1["rank_type_v2"].value_counts()

C:\Users\aikuy\AppData\Local\Temp\ipykernel_34604\4260155846.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


rank_type_v2
rank_none    117994
rank_game     86430
Name: count, dtype: int64

In [165]:
#레벨 별로 나누어 랭크게임 가능/불가능 여부 판별

df1["account_level_group"] = df1["accountLevel"].apply(
    lambda x: "over 30" if x >= 30 else "under 30"
)


normal_game_df = df1[df1["rank_type_v2"] == "rank_none"]

normal_game_count = (
    normal_game_df
    .groupby("account_level_group")
    ["gameId"]
    .nunique()
    .reset_index(name="normal_game_count")
)

normal_game_count

,account_level_group,normal_game_count
0,over 30,6167
1,under 30,5181


In [166]:
# 계정 레벨 그룹 분포
df1["account_level_group"].value_counts()

account_level_group
over 30     178134
under 30     26290
Name: count, dtype: int64

In [167]:
# 랭크가 아닌 게임만 뽑아서 계산
count_v2 = (
    df1[df1["rank_type_v2"] == "rank_none"]
    .groupby(["matchingTeamMode", "account_level_group"])
    .size()
    .reset_index(name="game_count")
)

count_v2

,matchingTeamMode,account_level_group,game_count
0,3,over 30,80533
1,3,under 30,25773
2,4,over 30,11171
3,4,under 30,517


In [168]:
df1[df1['account_level_group'] == 'over 30']['rank_type_v2'].value_counts()

rank_type_v2
rank_none    91704
rank_game    86430
Name: count, dtype: int64

In [169]:
# 팀모드, 계정렙, 랭크게임 여부에 따른 분포
play_count = (
    df1
    .groupby(
        ["matchingTeamMode", "account_level_group", "rank_type_v2"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "play_count"})
)

play_count

,matchingTeamMode,account_level_group,rank_type_v2,play_count
0,3,over 30,rank_game,86430
1,3,over 30,rank_none,80533
2,3,under 30,rank_none,25773
3,4,over 30,rank_none,11171
4,4,under 30,rank_none,517


In [170]:
# 이름 비슷한 컬럼 같은 지 여부 판별
df1["isLeavingBeforeCreditRevivalTerminate"].equals(df1["IsLeavingBeforeCreditRevivalTerminate"])

True

In [171]:
df1["isLeavingBeforeCreditRevivalTerminate"].value_counts()

isLeavingBeforeCreditRevivalTerminate
False    203826
True        598
Name: count, dtype: int64

In [172]:
df1["giveUp"].value_counts()

giveUp
0    203447
1       977
Name: count, dtype: int64

In [173]:
required_columns = [
    # 기본 식별 및 세그먼트
    'accountLevel', 'gameId', 'userNum', 'serverName', 'gameRank', 'rankPoint', 'matchingTeamMode', 
    'versionMajor', 'versionMinor',
    
    # 6대 KPI 계산용 (승률, KDA, DPM, 생존시간)
    'victory', 'playerKill', 'playerAssistant', 'playerDeaths', 
    'damageToPlayer', 'damageFromPlayer', 'playTime', 'survivableTime',
    
    # 팀 서포트 및 다양성 (회복, 보호막, 시야, 캐릭터)
    'teamRecover', 'protectAbsorb', 'viewContribution', 'characterNum',
    
    # 자원 전환 및 성장 효율
    'totalGainVFCredit', 'totalUseVFCredit',  # 총 획득/사용 크레딧
    'transferConsoleFromRevivalUseVFCredit', # 부활에 쓴 크레딧(안 필요할 수도 근데 그냥 제 경험 기반으로 넣어놓음)
    'transferConsoleFromMaterialUseVFCredit', # 재료에 쓴 크레딧
    'craftEpic', 'craftLegend', 'craftMythic', # 제작 아이템 등급별 수량
    
    # 개인 추세 및 MMR
    'mmrGainInGame', 'mmrLossEntryCost',
    
    #
    'isLeavingBeforeCreditRevivalTerminate', 'giveUp', "account_level_group", "rank_type_v2","rank_group","totalTime","watchTime"
]

# 더 필요한 컬럼이 있다면 추후 추가하면 됨
# 태블로에서 데이터-새로고침 하면 됩니다

core_df = df1[required_columns].copy()

core_df

,accountLevel,gameId,userNum,serverName,gameRank,rankPoint,matchingTeamMode,versionMajor,versionMinor,victory,...,craftMythic,mmrGainInGame,mmrLossEntryCost,isLeavingBeforeCreditRevivalTerminate,giveUp,account_level_group,rank_type_v2,rank_group,totalTime,watchTime
0,119,35260427,4378745,Seoul,5,5634,3,22,1,0,...,0,30,-44,False,0,over 30,rank_game,플래티넘&다이아,1159,34
1,82,35260427,4313007,Seoul,4,5625,3,22,1,0,...,0,69,-44,False,0,over 30,rank_game,플래티넘&다이아,1220,0
2,133,35260427,4227659,Seoul,2,5285,3,22,1,0,...,0,128,-43,False,0,over 30,rank_game,플래티넘&다이아,1342,197
3,156,35260427,3979035,Seoul,1,5316,3,22,1,1,...,0,124,-43,False,0,over 30,rank_game,플래티넘&다이아,1342,0
4,150,35260427,2118488,Seoul,1,5439,3,22,1,1,...,0,124,-43,False,0,over 30,rank_game,플래티넘&다이아,1342,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204419,10,35765219,3927798,Europe,3,0,3,23,0,0,...,0,0,0,True,1,under 30,rank_none,아이언&브론즈,855,17
204420,16,35783095,4932642,Asia,7,0,3,23,0,0,...,0,0,0,True,1,under 30,rank_none,아이언&브론즈,147,0
204421,118,35796912,456574,Asia,8,0,3,23,0,0,...,0,0,0,True,1,over 30,rank_none,아이언&브론즈,90,0
204422,56,35804415,4519640,Asia,8,4368,3,23,0,0,...,0,0,-72,True,1,over 30,rank_game,플래티넘&다이아,48,0


In [174]:
# 수치 통일 위해 정규화
core_df['scoreRecover'] = df1['teamRecover'] / df1['teamRecover'].replace(0, 1).max()
core_df['scoreProtect'] = df1['protectAbsorb'] / df1['protectAbsorb'].replace(0, 1).max()
core_df['scoreVision'] = df1['viewContribution'] / df1['viewContribution'].replace(0, 1).max()
core_df['scoreCreditRevive'] = df1['creditRevivedOthersCount'] / df1['creditRevivedOthersCount'].replace(0, 1).max()

{'rdKDA': np.float64(0.7355988603962178), 'rdDPM': np.float64(0.9104335795080932), 'rdSurvival_Time': np.float64(0.6018509041539606), 'rdCombat_Density': np.float64(0.6544880033560248), 'rdTeam_Support': np.float64(0.5894683203504072), 'rdDamage_Ratio': np.float64(0.33536848565935956)}


In [177]:
#core_df 최종 저장 
core_df.to_csv('ER_core_data22.csv', index=False, encoding='utf-8-sig')

In [178]:
core_df.groupby("account_level_group")["rdKDA"].mean()

# 랭크 3.59~3.97
# 비랭크 3.68~3.86


account_level_group
over 30     3.682952
under 30    3.868360
Name: rdKDA, dtype: float64

In [179]:
core_df.groupby("rank_group")["rdDPM"].value_counts()
#랭크  683~764
#비랭크 534~782

C:\Users\aikuy\AppData\Local\Temp\ipykernel_34604\1955235378.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  core_df.groupby("rank_group")["rdDPM"].value_counts()


rank_group  rdDPM      
아이언&브론즈     0.000000       294
            360.000000      18
            540.000000      17
            660.000000      13
            240.000000      12
                          ... 
메테오라이트+     5538.933902      0
            5687.786885      0
            5850.332594      0
            6145.476603      0
            6320.996068      0
Name: count, Length: 797468, dtype: int64

In [180]:
core_df.groupby("account_level_group")["rdSurvival_Time"].mean()
#랭크 27.3~29.4
#비랭크 27.6~27.8

account_level_group
over 30     27.855957
under 30    27.607950
Name: rdSurvival_Time, dtype: float64

In [181]:
core_df.groupby("account_level_group")["rdCombat_Density"].mean()

# 랭크 0.34~0.44
# 비랭크 0.32~0.42

account_level_group
over 30     0.426392
under 30    0.327098
Name: rdCombat_Density, dtype: float64

In [182]:
core_df.groupby("account_level_group")["rdTeam_Support"].mean()

# 랭크 0.33~0.41
# 비랭크 0.27~0.36

account_level_group
over 30     0.358072
under 30    0.274833
Name: rdTeam_Support, dtype: float64

In [183]:
core_df.groupby("rank_group")["rdDamage_Ratio"].mean()

#비랭크 1.05~1.43
#랭크 1.00~1.12

C:\Users\aikuy\AppData\Local\Temp\ipykernel_34604\3078348203.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  core_df.groupby("rank_group")["rdDamage_Ratio"].mean()


rank_group
아이언&브론즈     1.129685
실버&골드       1.002861
플래티넘&다이아    1.045698
메테오라이트+     1.079956
Name: rdDamage_Ratio, dtype: float64

In [184]:
# give Up과 isLeaving... 중 하나라도 만족하는 게임 수 뽑아낼 예정